# 17 · Augmentation aimed at the known failure (Phase 3)

**Goal.** Notebooks 14, 15 and 16b established what the detector gets wrong: it flags careful, fluent students, and within a prompt the flagged students write **more predictable** text (GPT-2 perplexity −0.95 SD), with **fewer misspellings** (−0.43 SD) and more AI-typical vocabulary, not longer essays. Two cheap training-time changes target exactly that, and this notebook tests whether they lower false flags on topics the model has never seen.

- **Variant A, spelling/typo augmentation.** The model uses misspellings as "human" evidence, so make misspellings uninformative: repair the spelling of half of the human training essays, and add human-like typos to half of the AI training essays.
- **Variant B, predictability-matched weighting.** Predictability is partly a real AI signal but also a shortcut: weight the loss so that human and AI essays carry equal weight in every GPT-2-perplexity bin.
- **Variant C = A + B**, run only if A and B each lower false flags on their own.

Everything below up to the "Pre-registered decision rule" was written **before any variant was trained**, and committed to git before the first run, so the rule cannot drift to fit the results.

## 1 · Design, fixed before anything runs

**The seven left-out prompts** (the leave-one-prompt-out subset). Full leave-one-prompt-out over 13 prompts costs 3.4 hours per variant, so each variant is run on seven folds chosen **by a rule applied to the notebook 16b results (the original recipe, 1% threshold)**: the four worst prompts, the median prompt, and the two best. The two held-out prompts (Summer projects, Electoral college) play no part in any choice and are not scored in this phase; held-out generators are not scored either. They are checked once, at the end of Phase 5, for the final model, threshold and ensemble together.

**One seed per fold per variant: seed 42, the same as the baseline folds.** Notebook 16b measured seed noise at 0.34 points of false-flag rate on one fold, well below the effects we care about. Each variant is compared prompt by prompt, and paired, with the saved original-recipe fold scores (`results/16b_lopo/scores_foldKK_seed42.csv`).

**Training recipe: identical to notebook 08** (2 epochs, lr 2e-5, 6% warm-up, 512 tokens, fp16, batch 16, best epoch by val AUC of the other-12 prompts). Only the training data (A) or the loss weights (B) change. Val and test essays, the left-out prompt's essays and all held-out text are **never modified**.

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.utils import simple_preprocess
from sklearn.metrics import roc_auc_score

from src.data import load_splits

SEED = 42
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120)
HUMAN, AI = '#2a78d6', '#eb6834'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e6e3', 'grid.linewidth': 0.6, 'axes.edgecolor': '#9a9a94', 'axes.titleweight': 'bold'})

log16 = pd.read_csv('../results/16b_lopo/log.csv')
base_log = log16[log16['seed'] == 42].drop_duplicates('fold').sort_values('fpr_pct', ascending=False).reset_index(drop=True)
median_fold = int(base_log.loc[(base_log['fpr_pct'] - base_log['fpr_pct'].median()).abs().idxmin(), 'fold'])
RULE = list(base_log['fold'][:4]) + [median_fold] + list(base_log['fold'][-2:])
FOLDS = [int(f) for f in RULE]
sel = base_log[base_log['fold'].isin(FOLDS)].copy()
sel['role'] = [('worst four' if f in RULE[:4] else 'median' if f == median_fold else 'best two') for f in sel['fold']]
print('FOLDS (in the order the runs will use):', FOLDS)
sel[['fold', 'left_out_prompt', 'role', 'fpr_pct', 'fpr_lo', 'fpr_hi']].rename(columns={'fpr_pct': 'baseline humans flagged % (1%)'})

FOLDS (in the order the runs will use): [11, 4, 3, 6, 7, 12, 2]


,fold,left_out_prompt,role,baseline humans flagged % (1%),fpr_lo,fpr_hi
0,11,Seeking multiple opinions,worst four,5.80,4.74,7.07
1,4,Distance learning,worst four,5.52,4.63,6.56
2,3,Community service,worst four,5.19,4.19,6.41
3,6,Exploring Venus,worst four,4.73,3.85,5.79
6,7,Facial action coding system,median,2.54,1.96,3.29
11,12,The Face on Mars,best two,0.57,0.30,1.08
12,2,Cell phones at school,best two,0.36,0.17,0.79


## 2 · Pre-registered decision rule

*(Written before any variant was trained.)*

For each variant, the same seven folds, the same seed (42), the same left-out essays as the baseline. For each fold, a model's threshold is the score that flags 1% of the **val humans of the other 12 prompts**, as in notebooks 09 and 16b.

**Primary statistic.** The **mean, over the seven prompts, of the left-out-prompt human false-flag rate** at that threshold, as a **paired difference** (variant minus original recipe). The 95% interval is a bootstrap (1,000 resamples, seed 42) that redraws, in each fold, the left-out humans (whole `dup_cluster` groups, the same essays for both models) *and* the val humans the threshold is chosen on.

**Secondary statistics (reported, not used to decide):**

1. the **worst-prompt** false-flag rate (the maximum over the seven folds), variant against baseline;
2. the false-flag rate among the **most predictable quarter of human essays**: left-out humans whose GPT-2 log-perplexity is below the 25th percentile of the human *training* essays (the cutoff is computed in section 4 below), pooled over the seven prompts.

**Guardrail.** The mean over the seven folds of **val AI caught at the 1% threshold** (the share of AI essays in the other-12-prompt val set that score above the threshold) must **not drop by more than 0.5 percentage points** against the baseline.

**A variant wins** if and only if the primary interval **excludes zero on the negative side** and the **guardrail holds**.

- **Variant C** (A + B) is run only if the primary *point estimates* of A and B are each below zero. It is judged by the same rule, against the baseline.
- If more than one variant wins, the winner is the one with the most negative primary point estimate.
- **Then:** the winner (or, if nothing wins, the original recipe) is trained on the full training set with **3 seeds** (42, 43, 44) and saved to new model folders under `models/`, for Phase 5's ensemble. No existing file in `models/` or `results/` is overwritten.
- No further change is made to a variant after seeing its result. If a variant fails, it is reported as failed.

## 3 · Variant A: the spelling repair and the typo injection

### The spell-checker (named before use)

There is no spell-checker installed on this machine and adding a dependency needs your approval, so the corrector is built from what the project already has: a **Norvig-style corrector on the GloVe vocabulary** (`src/spelling.py`; GloVe-100 from `~/gensim-data`, offline).

- A word is a candidate for repair only if it is **not in the 400,000-word GloVe vocabulary**. That is exactly the test notebook 14 used to *measure* misspellings, so "misspelled" means the same thing when measured and when repaired.
- Candidates are the words one letter-edit away (swap, delete, replace, insert) among the alphabetic GloVe words; transpositions are tried first ("casue" → "cause"), then candidates that keep the first letter, then the most frequent. Two edits are tried only for words of eight or more letters with no one-edit neighbour, only among the 100,000 most frequent words, and only if they keep the first letter. (A first version limited one-edit candidates to the 100,000 most frequent words and turned "driveless" into "priceless"; it was widened after reading the examples and before any model was trained.)
- Words under 4 letters, all-capital words (acronyms), capitalised words in the middle of a sentence (names), and words with no candidate are left alone. Contractions typed without an apostrophe ("dont", "shouldnt") are mapped to the proper form ("don't", "shouldn't") except where the bare form is a real word ("were", "its", "well").
- **Why this and not a package:** no new dependency, works offline, and it is the same vocabulary as the measurement. **Its limits** are shown by the examples below: roughly 85–90% of the corrections are right on a hand-read sample, and the misses are real words absent from the vocabulary or rare words ("driveless" → "priceless").

### The augmentation

Seeded per essay (from `(42, row_id)`), so the result does not depend on the fold:

- **Human training essays:** with probability 0.5, repair the spelling.
- **AI training essays:** with probability 0.5, inject typos (`inject_typos`: swap, delete or double one letter in a share of the words) at a rate drawn at random from the **real distribution of human misspelling rates** in the training essays, so injected typos look like students' typos in amount.

After this, about half of each class has the human-like misspelling profile and half has none, so the misspelling rate carries (almost) no information about the label. The training set keeps its size, so a fold still takes about 15 minutes.

In [2]:
import gensim.downloader as gensim_api
from src.spelling import SpellCorrector
from src.augment import build_augmented_train
from src.paraphraser import inject_typos

t0 = time.time()
kv = gensim_api.load('glove-wiki-gigaword-100')
corrector = SpellCorrector(kv=kv)
glove_vocab = corrector.vocab
print(f'corrector built in {time.time() - t0:.0f}s')

data = load_splits()
train = data[data['split'] == 'train']
gpt2 = pd.read_csv('../results/10_gpt2_features.csv', index_col='row_id')
train = train.join(gpt2[['log_ppl']])
print(f"train: {len(train):,} essays, {(train['label'] == 0).sum():,} human, {(train['label'] == 1).sum():,} AI")

def misspell_pct(text):
    toks = simple_preprocess(text, min_len=1, max_len=30)
    return 100 * np.mean([w not in glove_vocab for w in toks] or [0])

t0 = time.time()
train['misspell_pct'] = [misspell_pct(t) for t in train['text']]
print(f'misspelling measure for all train essays in {time.time() - t0:.0f}s')
hum_rates = train.loc[train['label'] == 0, 'misspell_pct'].to_numpy() / 100
print(f"human misspelling rate (share of tokens missing from GloVe): mean {hum_rates.mean() * 100:.2f}%, median {np.median(hum_rates) * 100:.2f}%, "
      f"90th percentile {np.percentile(hum_rates, 90) * 100:.2f}%; AI mean {train.loc[train['label'] == 1, 'misspell_pct'].mean():.2f}%")
print(f"AUC of the misspelling rate alone for separating human (0) from AI (1) on train: {roc_auc_score(train['label'], train['misspell_pct']):.3f}  (0.5 = no information; below 0.5 = humans misspell more)")

corrector built in 84s


train: 28,006 essays, 18,325 human, 9,681 AI


misspelling measure for all train essays in 18s
human misspelling rate (share of tokens missing from GloVe): mean 1.05%, median 0.54%, 90th percentile 2.80%; AI mean 0.08%
AUC of the misspelling rate alone for separating human (0) from AI (1) on train: 0.179  (0.5 = no information; below 0.5 = humans misspell more)


### Calibrating the typo rate

`inject_typos` edits a share `rate` of the words, but some edits turn a word into another real word ("form" → "from"), which the misspelling measure does not count. So the injected rate is scaled by one factor so that the *measured* misspelling rate of an augmented AI essay matches the human one. The factor is estimated on 600 AI training essays (training essays only; no model is involved).

In [3]:
rng = np.random.default_rng(SEED)
probe = train[train['label'] == 1].sample(600, random_state=SEED)
rates = rng.choice(hum_rates, len(probe))
measured = np.array([misspell_pct(inject_typos(t, r, seed=i)) / 100 for i, (t, r) in enumerate(zip(probe['text'], rates))])
RATE_FACTOR = float(rates.mean() / measured.mean())
print(f"asked-for rate {rates.mean() * 100:.2f}% -> measured {measured.mean() * 100:.2f}%   => scale the injected rate by {RATE_FACTOR:.2f}")

asked-for rate 1.04% -> measured 0.59%   => scale the injected rate by 1.75


In [4]:
CACHE = '../data/external/17_augmented_train.csv'
os.makedirs('../data/external', exist_ok=True)
if os.path.exists(CACHE):
    aug = pd.read_csv(CACHE, index_col='row_id')
    print('loaded', CACHE)
else:
    t0 = time.time()
    aug = build_augmented_train(train, corrector, hum_rates, share=0.5, rate_factor=RATE_FACTOR, seed=SEED)
    aug.to_csv(CACHE)
    print(f'augmented {len(aug):,} essays in {time.time() - t0:.0f}s -> {CACHE} (git-ignored; a derived copy of training text)')
assert aug.index.equals(train.index)
print(aug.groupby([train['label'].map({0: 'human', 1: 'AI'}), aug['kind']]).size().unstack().fillna(0).astype(int))
train['text_aug'] = aug['text_aug'].to_numpy()
t0 = time.time()
train['misspell_pct_aug'] = [misspell_pct(t) for t in train['text_aug']]
print(f'measured again in {time.time() - t0:.0f}s')

augmented 28,006 essays in 165s -> ../data/external/17_augmented_train.csv (git-ignored; a derived copy of training text)
kind   none  spell_fixed  typos
label                          
AI     4853            0   4828
human  9260         9065      0


measured again in 18s


### Before and after

In [5]:
import difflib, re
def show_changes(i, k=4, ctx=45):
    r = train.loc[i]
    a, b = r['text'], r['text_aug']
    sm = difflib.SequenceMatcher(None, a.split(), b.split(), autojunk=False)
    ops = [op for op in sm.get_opcodes() if op[0] != 'equal'][:k]
    wa, wb = a.split(), b.split()
    out = []
    for tag, i1, i2, j1, j2 in ops:
        left = ' '.join(wa[max(0, i1 - 5):i1]); right = ' '.join(wa[i2:i2 + 5])
        out.append(f"   ...{left} [{' '.join(wa[i1:i2])}  ->  {' '.join(wb[j1:j2])}] {right}...")
    return out

rng = np.random.default_rng(1)
for label, kind, name in [(0, 'spell_fixed', 'HUMAN essay, spelling repaired'), (1, 'typos', 'AI essay, typos injected')]:
    pool = train[(train['label'] == label) & (aug['kind'] == kind) & (aug['n_changed'] >= 3)]
    for i in rng.choice(pool.index, 3, replace=False):
        print(f"[{name} | {train.loc[i, 'source']} | {train.loc[i, 'prompt_name']} | measured misspelling {train.loc[i, 'misspell_pct']:.1f}% -> {train.loc[i, 'misspell_pct_aug']:.1f}%]")
        print('\n'.join(show_changes(i))); print()

[HUMAN essay, spelling repaired | persuade_corpus | Facial action coding system | measured misspelling 0.6% -> 0.0%]
   ...muscles. without observation of this [mucles  ->  muscles] movement in the body the...
   ...to calculate like humans. in [anothert  ->  another] different way the computers have...
   ...raise your lips at the [cornersof  ->  corners] your mouth, then squint your...

[HUMAN essay, spelling repaired | persuade_corpus | Facial action coding system | measured misspelling 2.9% -> 0.0%]
   ...good information about the new [inovation  ->  innovation] to emotional technology. As the...
   ...are going to have new [opertunities  ->  opportunities] such as this one to...
   ...appears on your screen, a [smililar  ->  similar] ad might follow. But if...
   ...can the camera see? Humans [relie  ->  reile] in the enternet to bank...

[HUMAN essay, spelling repaired | persuade_corpus | Driverless cars | measured misspelling 1.7% -> 0.4%]
   ...that, "Presently, traffic laws a

In [6]:
# 60 random human-essay corrections, to judge the corrector's accuracy by eye
import random
pairs = []
for i in train[(train['label'] == 0) & (aug['kind'] == 'spell_fixed')].sample(300, random_state=3).index:
    pairs += corrector.correct_text(train.loc[i, 'text'])[2]
random.Random(7).shuffle(pairs)
print(len(pairs), 'corrections in 300 essays; 60 random ones:')
print(', '.join(f'{a}->{b}' for a, b in pairs[:60]))

1120 corrections in 300 essays; 60 random ones:
derection->direction, crawn->crown, Reasearchers->Researchers, unbareable->unbearable, thats->that's, appered->appeared, driveless->driverless, angery->angry, benefical->beneficial, trafficed->trafficked, distination->destination, istics->itics, cemters->centers, equipemant->equipement, discribe->describe, probly->proby, becomeing->becoming, instensive->intensive, apperared->appreared, tehcnological->technological, cliam->claim, polluants->pollutants, usig->using, builing->building, balistic->ballistic, Ithink->Think, dricers->drivers, becouse->because, scientest->scientist, rhetoricle->rhetoric, dont->don't, opprotunity->opportunity, desities->deities, wont->won't, opoinions->opinions, beleve->believe, emmisions->emmissions, cafetiera->cafeteria, charcter->character, evrything->everything, dont->don't, arent->aren't, scientis->scientist, sciencetists->scientists, situatiion->situation, im->i'm, briveles->bristles, wouldnt->wouldn't, idal

### Does misspelling still tell the classes apart?

The aim of Variant A is to remove the misspelling shortcut. Two direct checks on the training essays: the AUC of the misspelling rate alone for separating human from AI (0.5 means no information), and the distribution of the misspelling rate in each class, before and after.

In [7]:
auc_b = roc_auc_score(train['label'], train['misspell_pct']); auc_a = roc_auc_score(train['label'], train['misspell_pct_aug'])
print(f"AUC of the misspelling rate alone (human vs AI): before {auc_b:.3f}  ->  after {auc_a:.3f}   (0.5 = no information)")
q = lambda s: pd.Series({'mean %': s.mean(), 'median %': s.median(), '75th pct': s.quantile(.75), '90th pct': s.quantile(.90), 'share with 0%': (s == 0).mean() * 100})
tab = pd.concat({'human, before': q(train.loc[train['label'] == 0, 'misspell_pct']), 'human, after': q(train.loc[train['label'] == 0, 'misspell_pct_aug']),
                 'AI, before': q(train.loc[train['label'] == 1, 'misspell_pct']), 'AI, after': q(train.loc[train['label'] == 1, 'misspell_pct_aug'])}, axis=1)
print(tab.round(2).to_string())

AUC of the misspelling rate alone (human vs AI): before 0.179  ->  after 0.428   (0.5 = no information)
               human, before  human, after  AI, before  AI, after
mean %                  1.05          0.63        0.08       0.54
median %                0.54          0.20        0.00       0.00
75th pct                1.47          0.75        0.00       0.56
90th pct                2.80          1.85        0.00       1.73
share with 0%          29.22         45.99       90.46      63.46


## 4 · Variant B: predictability-matched weights

**What it does.** Each training essay has a GPT-2 log-perplexity (computed in notebook 10; lower means a more predictable text). The training essays are cut into ten bins by that value. In every bin the loss of the human essays and the loss of the AI essays are weighted so that *both classes carry the same total weight*, i.e. the human and AI weighted distributions over the bins become the same. Class totals are kept (mean weight 1 per class), so the class balance is unchanged.

**Why it targets the 16b finding.** False flags are the students whose text is as predictable as AI text. In the raw training data a predictable essay is almost always AI (99.8% AI in the most predictable decile), so "predictable" is nearly a label by itself and the cheapest thing for the network to learn. Reweighting makes a predictable human essay count as much as the many predictable AI essays, so predictability alone stops paying and the network has to find other cues.

**Why it might cost detection.** Predictability *is* a real signal of AI writing (notebook 10: GPT-2 perplexity alone catches AI with AUC 0.98 on test). Down-weighting the very predictable AI essays and up-weighting the rare unpredictable AI essays could weaken the model on exactly the AI essays that are easiest to catch, and the weights are noisy where one class is almost absent in a bin. The guardrail (val AI caught within 0.5 points) exists to catch that.

**The design choice, fixed now.** Ten bins by deciles; ideal weights are clipped to [0.1, 10] and each class is rescaled to mean weight 1. Without the clip the ideal weights reach 262× for the 7 human essays in the most predictable decile and 322× for the 3 AI essays in the least predictable (too noisy to train on). Four settings (bins 5 or 10, clip 5, 10 or 20) were compared **only by the weight distribution** below, never by a model result; ten bins with clip 10 sits in the middle.

In [8]:
from src.augment import predictability_weights, bin_distribution_gap
w, b = predictability_weights(train['log_ppl'], train['label'], n_bins=10, max_weight=10.0)
t = pd.crosstab(b, train['label']); t.columns = ['human', 'AI']
t['AI share %'] = 100 * t['AI'] / (t['human'] + t['AI'])
t['mean weight, human'] = pd.Series(w[train['label'].to_numpy() == 0]).groupby(b[train['label'].to_numpy() == 0]).mean()
t['mean weight, AI'] = pd.Series(w[train['label'].to_numpy() == 1]).groupby(b[train['label'].to_numpy() == 1]).mean()
t.index.name = 'predictability decile (0 = most predictable)'
print(t.round(2).to_string())
print(f"\ndistance between the human and AI distributions over the bins (0 = identical, 1 = disjoint): "
      f"raw {bin_distribution_gap(np.ones(len(w)), b, train['label'], 10):.3f} -> weighted {bin_distribution_gap(w, b, train['label'], 10):.3f}")
print(f"effective sample size {w.sum() ** 2 / (w ** 2).sum():,.0f} of {len(w):,} essays; weights range {w.min():.2f} to {w.max():.2f}")
CUTOFF = float(train.loc[train['label'] == 0, 'log_ppl'].quantile(0.25))
print(f"\nthe 'most predictable quarter of human essays' (secondary statistic) = GPT-2 log-perplexity below {CUTOFF:.4f} (25th percentile of the human training essays)")

                                              human    AI  AI share %  mean weight, human  mean weight, AI
predictability decile (0 = most predictable)                                                              
0                                                 7  2794       99.75               12.17             0.48
1                                                33  2767       98.82               12.17             0.48
2                                               396  2405       85.86                5.63             0.56
3                                              1743  1057       37.75                1.28             1.27
4                                              2432   369       13.17                0.92             3.63
5                                              2630   170        6.07                0.85             7.88
6                                              2727    74        2.64                0.82            13.83
7                                    

## 5 · Run time

Each fold trains on about 26,000 essays and takes about the same time as the baseline folds of notebook 16b.

In [9]:
mins = log16['minutes'].mean()
est = pd.DataFrame({'folds': [len(FOLDS)] * 3 + [3], 'hours': [len(FOLDS) * mins / 60] * 3 + [3 * mins * 28006 / 26000 / 60]},
                   index=['Variant A (7 folds)', 'Variant B (7 folds)', 'Variant C = A + B (7 folds, only if A and B both lower false flags)', 'final model, 3 seeds on the full training set'])
est.loc['A + B together'] = [2 * len(FOLDS), 2 * len(FOLDS) * mins / 60]
print(f'baseline folds took {mins:.1f} minutes each (notebook 16b)')
est.round(2)

baseline folds took 15.2 minutes each (notebook 16b)


,folds,hours
Variant A (7 folds),7.0,1.77
Variant B (7 folds),7.0,1.77
"Variant C = A + B (7 folds, only if A and B both lower false flags)",7.0,1.77
"final model, 3 seeds on the full training set",3.0,0.82
A + B together,14.0,3.54
